# Use Cache Demo

本文档演示如何使用 Nexus 的缓存功能进行录制、回放和故障恢复。 我们将通过一个完整的示例展示：

1. 使用真实注册的 `demo_agent` 和 `identity` task
2. 使用 `mock` OpenAI client 模拟 LLM 调用和错误
3. 模拟运行中途报错的场景
4. 使用缓存从断点恢复继续运行

示例包含两种使用方式：
- **编程方式**：使用 Runner API
- **命令行方式**：使用 `nexus run` 命令

## 初始化环境

首先导入必要的模块并配置日志。

In [1]:
import tempfile
from pathlib import Path

from nexus.utils.log import setup_logging

setup_logging()

## Mock OpenAI Client 介绍

Nexus 提供了一个 `mock` OpenAI client，用于测试和演示场景。 它支持：

- **预定义响应序列**：按顺序返回指定的响应
- **内联异常**：在 responses 中直接定义异常，实现任意位置的失败模拟
- **循环使用**：响应序列用完后会从头开始循环

配置示例：
```python
openai_clients={
    'main': {
        'name': 'mock',
        'config': {
            'client_args': {
                'responses': [
                    '第一次响应',
                    {'exception': 'TimeoutError', 'message': 'API 超时'},  # 第2次失败
                    '第三次响应（恢复后）',
                ],
            },
        },
    },
}
```

## 场景一：正常运行并缓存

首先演示一个正常运行的场景，LLM 调用会被缓存。

In [2]:
from nexus.runner.single import RunInput, CacheConfig, run
from nexus.utils.cache import load_cache, get_cache_stats

# 创建缓存文件路径
cache_file = Path(tempfile.gettempdir()) / "cache_demo_normal.pkl"

# 配置运行输入
run_input_normal = RunInput(
    imports=['nexus.extensions', 'nexus.extensions.tasks.identity'],
    task_name='identity',
    task_config={},
    agent_name='demo_agent',
    agent_config={
        'user_prompt': '请简单介绍一下自己',
        'max_iterations': 3,
    },
    # 使用 mock client，直接返回预定义响应
    openai_clients={
        'main': {
            'name': 'mock',
            'config': {
                'client_args': {
                    'responses': [
                        '你好！我是一个 AI 助手，很高兴为你服务。',
                    ],
                },
            },
        },
    },
    cache=CacheConfig(
        enabled=True,
        file_path=str(cache_file),
    ),
)

print(f"缓存文件: {cache_file}")
print(f"Task: {run_input_normal.task_name}")
print(f"Agent: {run_input_normal.agent_name}")

缓存文件: /tmp/cache_demo_normal.pkl
Task: identity
Agent: demo_agent


In [3]:
def get_last_assistant_message(agent_output):
    """从 agent_output 中提取最后一条 assistant 消息。"""
    if agent_output is None:
        return None
    # demo_agent 使用 main_trajectory 存储对话历史
    trajectory = getattr(agent_output, 'main_trajectory', None)
    if trajectory:
        for msg in reversed(trajectory):
            if isinstance(msg, dict) and msg.get('role') == 'assistant' and msg.get('content'):
                return msg['content']
    return None

def print_cache_stats(stats):
    """打印缓存统计信息。"""
    if stats:
        print(f"  replay_hits: {stats.get('replay_hits', 0)} (从缓存读取)")
        print(f"  new_cached: {stats.get('new_cached', 0)} (新缓存)")
        print(f"  total: {stats.get('total', 0)} (缓存条目总数)")

# 第一次运行
print("=== 第一次运行 ===")
output1 = await run(run_input_normal)

print(f"\n运行结果:")
print(f"  成功: {output1.success}")
print_cache_stats(output1.cache_stats)
last_msg = get_last_assistant_message(output1.agent_output)
if last_msg:
    print(f"  Agent 最终响应: {last_msg}")

=== 第一次运行 ===
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions' with 77 registrations:
2026-01-27 03:54:45 | INFO | nexus.core.registry |   - [agent] browser_gym_agent, claude_code, demo_agent, toolcall_agent, swe_agent, rondo_agent, terminus2, calculator_agent
2026-01-27 03:54:45 | INFO | nexus.core.registry |   - [tool] browser_gym/submit_result, base/write_todos, base/finish, demo/think, demo/bash, swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit, calculator/calculate, browser/goto, b [... (truncated log)] rowser/select_option, browser/new_tab, browser/close_tab, browser/focus_tab, browser/go_back, browser/go_forward, browser/get_observation, python_code_interpreter, google_search, hunyuan_search, visit
2026-01-27 03:54:45 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, brow

In [4]:
# 第二次运行（从缓存读取）
print("=== 第二次运行（从缓存）===")
output2 = await run(run_input_normal)

print(f"\n运行结果:")
print(f"  成功: {output2.success}")
print_cache_stats(output2.cache_stats)

# 验证两次结果一致
msg1 = get_last_assistant_message(output1.agent_output)
msg2 = get_last_assistant_message(output2.agent_output)
if msg1 and msg2:
    print(f"\n两次响应一致: {msg1 == msg2}")

=== 第二次运行（从缓存）===
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions.tasks.identity' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.runner.single | Starting runner for task: identity (run_id: bb559cd4c714)
2026-01-27 03:54:45 | INFO | nexus.utils.cache | Loaded cache with 1 top-level entries from /tmp/cache_demo_normal.pkl
2026-01-27 03:54:45 | INFO | nexus.runner.single | Cache enabled with 1 existing entries
2026-01-27 03:54:45 | INFO | nexus.runner.single | Executing task: identity
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Model output: 你好！我是一个 AI 助手，很高兴为你服务。
2026-01-27 03:54:45 | INFO | nexus.runner.single | Task completed: identity
2026-01-27 03:54:45 | INFO | nexus.utils.cache | Saved cache with 1 top-level entries to /tmp/cache_demo_normal.pkl

运行结果:
  成功: True
  replay_hits: 1 (从缓存读取)
  new_cached: 0 (新缓存)
  to

## 场景二：运行中途报错，然后恢复

这是缓存功能最重要的应用场景：当 Agent 执行到一半报错时，已成功的 LLM 调用会被缓存。 修复问题后重新运行，会从缓存读取已成功的调用，只重新执行失败的部分。

### 模拟场景

1. **第一次运行**：模拟 LLM 服务不稳定，第2次调用失败
2. **第二次运行**：模拟 LLM 服务已恢复，使用相同的缓存文件重新运行

通过使用不同的 mock 配置来模拟"LLM 服务从故障中恢复"。

In [5]:
# 创建缓存文件路径
cache_file_fail = Path(tempfile.gettempdir()) / "cache_demo_fail.pkl"

# 删除旧缓存文件（如果存在）
if cache_file_fail.exists():
    cache_file_fail.unlink()

# 第一次运行的配置：模拟 LLM 服务不稳定
# 第1次调用成功（带 tool_call），第2次调用失败
run_input_fail = RunInput(
    imports=['nexus.extensions', 'nexus.extensions.tasks.identity'],
    task_name='identity',
    task_config={},
    agent_name='demo_agent',
    agent_config={
        'user_prompt': '请帮我计算 2 + 2',
        'max_iterations': 5,
    },
    openai_clients={
        'main': {
            'name': 'mock',
            'config': {
                'client_args': {
                    'responses': [
                        # 第1次调用：成功，带 tool_call（让 agent 继续循环）
                        {
                            'content': '让我思考一下这个问题...',
                            'tool_calls': [{
                                'id': 'call_1',
                                'function': {
                                    'name': 'Think',
                                    'arguments': '{"thought": "2 + 2 = 4"}',
                                },
                            }],
                        },
                        # 第2次调用：失败（模拟 LLM 服务故障）
                        {'exception': 'TimeoutError', 'message': '模拟的 API 超时错误'},
                    ],
                },
            },
        },
    },
    cache=CacheConfig(
        enabled=True,
        file_path=str(cache_file_fail),
    ),
)

print(f"缓存文件: {cache_file_fail}")
print("第一次运行配置（模拟故障）:")
print("  - 第1次调用: 成功（带 tool_call）")
print("  - 第2次调用: TimeoutError")

缓存文件: /tmp/cache_demo_fail.pkl
第一次运行配置（模拟故障）:
  - 第1次调用: 成功（带 tool_call）
  - 第2次调用: TimeoutError


In [6]:
# 第一次运行：会失败
print("=== 第一次运行（会失败）===")
output_fail = await run(run_input_fail)

print(f"\n运行结果:")
print(f"  成功: {output_fail.success}")
if output_fail.error:
    print(f"  错误类型: {output_fail.error.type}")
    print(f"  错误消息: {output_fail.error.message}")
print_cache_stats(output_fail.cache_stats)

=== 第一次运行（会失败）===
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions.tasks.identity' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.runner.single | Auto-generated run_id: 4ffea9f1e78d
2026-01-27 03:54:45 | INFO | nexus.runner.single | Starting runner for task: identity (run_id: 4ffea9f1e78d)
2026-01-27 03:54:45 | INFO | nexus.runner.single | Cache enabled with 0 existing entries
2026-01-27 03:54:45 | INFO | nexus.runner.single | Executing task: identity
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Model output: 让我思考一下这个问题...
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Tool call: Think({"thought": "2 + 2 = 4"})
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Tool output: [{'text': 'Thought recorded.', 'type': 'text'}]
2026-01-27 03:54:45 | ERROR | nexus.runner.single | Task failed: i

In [7]:
# 第二次运行的配置：模拟 LLM 服务已恢复
# 使用相同的缓存文件，但 mock 配置中只有成功响应
run_input_recover = RunInput(
    imports=['nexus.extensions', 'nexus.extensions.tasks.identity'],
    task_name='identity',
    task_config={},
    agent_name='demo_agent',
    agent_config={
        'user_prompt': '请帮我计算 2 + 2',
        'max_iterations': 5,
    },
    openai_clients={
        'main': {
            'name': 'mock',
            'config': {
                'client_args': {
                    'responses': [
                        # LLM 服务已恢复，所有调用都成功
                        '经过思考，2 + 2 = 4',
                    ],
                },
            },
        },
    },
    cache=CacheConfig(
        enabled=True,
        file_path=str(cache_file_fail),  # 使用相同的缓存文件
    ),
)

print("=== 第二次运行（LLM 服务已恢复）===")
print("说明: 第1次调用从缓存读取，第2次调用使用恢复后的 LLM 服务")

output_recover = await run(run_input_recover)

print(f"\n运行结果:")
print(f"  成功: {output_recover.success}")
print_cache_stats(output_recover.cache_stats)
last_msg = get_last_assistant_message(output_recover.agent_output)
if last_msg:
    print(f"  Agent 最终响应: {last_msg}")

=== 第二次运行（LLM 服务已恢复）===
说明: 第1次调用从缓存读取，第2次调用使用恢复后的 LLM 服务
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.core.registry | Imported 'nexus.extensions.tasks.identity' with no new registrations
2026-01-27 03:54:45 | INFO | nexus.runner.single | Auto-generated run_id: 466a09e385c3
2026-01-27 03:54:45 | INFO | nexus.runner.single | Starting runner for task: identity (run_id: 466a09e385c3)
2026-01-27 03:54:45 | INFO | nexus.utils.cache | Loaded cache with 2 top-level entries from /tmp/cache_demo_fail.pkl
2026-01-27 03:54:45 | INFO | nexus.runner.single | Cache enabled with 2 existing entries
2026-01-27 03:54:45 | INFO | nexus.runner.single | Executing task: identity
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Model output: 让我思考一下这个问题...
2026-01-27 03:54:45 | INFO | nexus.extensions.agents.demo.agent | Tool call: Think({"thought": "2 + 2 = 4"})
2026-01-27 03:54:45 | INFO | nexus.e

## 场景三：命令行使用（nexus run）

创建配置文件，然后使用 `nexus run` 命令执行。

In [8]:
# 创建配置文件 - 使用 mock client
config_content_mock = '''
imports:
  - nexus.extensions
  - nexus.extensions.tasks.identity

task_name: "identity"
task_config: {}

agent_name: "demo_agent"
agent_config:
  user_prompt: "请简单介绍一下自己"
  max_iterations: 3

# 启用缓存
cache:
  enabled: true
  file_path: /tmp/nexus_cache_cli.pkl

# 使用 mock client 进行测试
openai_clients:
  main:
    name: "mock"
    config:
      client_args:
        responses:
          - "你好！我是一个 AI 助手。"
'''

config_file_mock = Path(tempfile.gettempdir()) / "cache_demo_mock.yaml"
config_file_mock.write_text(config_content_mock)
print(f"Mock 配置文件已创建: {config_file_mock}")

Mock 配置文件已创建: /tmp/cache_demo_mock.yaml


### 使用 nexus run-single 命令执行

下面实际执行 `nexus run-single` 命令（使用 mock client，不需要真实 API）。

In [9]:
import subprocess

# 第一次执行命令
print("=== 第一次执行 nexus run-single ===")
result1 = subprocess.run(
    ['uv', 'run', 'nexus', 'run-single', str(config_file_mock)],
    capture_output=True,
    text=True,
)
print(result1.stdout)
if result1.returncode != 0:
    print(f"错误: {result1.stderr}")

# 第二次执行（从缓存读取）
print("\n=== 第二次执行 nexus run-single（从缓存）===")
result2 = subprocess.run(
    ['uv', 'run', 'nexus', 'run-single', str(config_file_mock)],
    capture_output=True,
    text=True,
)
print(result2.stdout)
if result2.returncode != 0:
    print(f"错误: {result2.stderr}")

=== 第一次执行 nexus run-single ===
2026-01-27 03:54:47 | INFO | nexus.core.registry | Imported 'nexus.extensions' with 77 registrations:
2026-01-27 03:54:47 | INFO | nexus.core.registry |   - [agent] browser_gym_agent, claude_code, demo_agent, toolcall_agent, swe_agent, rondo_agent, terminus2, calculator_agent
2026-01-27 03:54:47 | INFO | nexus.core.registry |   - [tool] browser_gym/submit_result, base/write_todos, base/finish, demo/think, demo/bash, swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit, calculator/calculate, browser/goto, b [... (truncated log)] rowser/select_option, browser/new_tab, browser/close_tab, browser/focus_tab, browser/go_back, browser/go_forward, browser/get_observation, python_code_interpreter, google_search, hunyuan_search, visit
2026-01-27 03:54:47 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browse

## 清理

清理临时文件。

In [10]:
# 清理临时文件
import os

files_to_clean = [
    cache_file,
    cache_file_fail,
    config_file_mock,
    Path('/tmp/nexus_cache_cli.pkl'),
]

for f in files_to_clean:
    if f.exists():
        os.remove(f)
        print(f"已删除: {f}")

print("\n清理完成")

已删除: /tmp/cache_demo_normal.pkl
已删除: /tmp/cache_demo_fail.pkl
已删除: /tmp/cache_demo_mock.yaml
已删除: /tmp/nexus_cache_cli.pkl

清理完成


## 总结

本文档展示了 Nexus 缓存功能的使用方式：

1. **使用真实 Agent**：`demo_agent` 是 Nexus 内置的演示 Agent
2. **使用 Mock Client**：`mock` OpenAI client 可以模拟 LLM 响应和错误
3. **故障恢复**：当运行中途失败时，已成功的调用会被缓存，重试时从缓存读取

缓存功能的主要优势：

- **故障恢复**：中途失败后可从断点继续
- **节省成本**：避免重复 LLM 调用
- **确定性测试**：使用录制的轨迹进行测试
- **调试支持**：重现问题场景